# Lab 06 · Image and NLP Enrichment (Notebook Walkthrough)

**Concept.** The `visual_nlp` profile adds `OcrSkill`, `ImageAnalysisSkill`, and `LanguageDetectionSkill` to the Blob skillset. This matters for a diagram-heavy engineering document where evidence can live in figures, not just prose.

**Where each signal lands (this is the part people get wrong):**
- `ImageAnalysisSkill` and `OcrSkill` run at the per-image context (`/document/normalized_images/*`), so their outputs land *under that path*. The indexer output field mappings must therefore read `/document/normalized_images/*/...` - and because `description` is a complex `{tags, captions}` object, the mapping drills into `captions/*/text` to capture the human-readable caption sentences. These populate the **Search-managed enrichment index** (`...-visual-nlp`), which **Agentic** retrieval can draw on.
- The app then merges a **document-level `image_description_text`** (the first several captions) back onto every **canonical chunk**, so `full_text` / `vector` / `hybrid` also gain a visual signal.
- The parser additionally attaches **figure thumbnails** (`image_evidence`) to the chunks on a figure's page, so the UI can show grounded visuals. Thumbnails are produced by **rendering each page and cropping the figure region** - not by pulling the raw embedded image. That matters because engineering PDFs often store figures as 1-bit image masks / soft-masked (SMask) stencils whose paint colour lives in the page content stream, so the raw bitmap is solid black; rendering composites them faithfully. The same render path also captures **vector-drawn charts** (analyst decks) that have no embedded image at all.

**Key lesson:** the caption *quality* depends on the document. Image Analysis 3.2 returns descriptive but generic captions (“a construction site with cranes and buildings”, “engineering drawing”), and OCR on vector-drawn diagrams yields fragmentary text (“17 07 2017”). Always **verify what actually landed** instead of assuming.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 0,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with image + NLP enrichment

In [2]:
job = lab.ingest(skill_profile='visual_nlp', reuse=True)
lab.chunk_overview(job)

Reusing existing 'visual_nlp' ingestion (doc_id=fa7dbea1, 414 chunks). Pass reuse=False to force a fresh run.


{'doc_id': 'fa7dbea16ef147788ed10128c9789d4b',
 'skill_profile': 'visual_nlp',
 'chunk_count': 414,
 'avg_tokens': 208.9,
 'max_tokens': 2092,
 'chunks_with_summary': 414,
 'chunks_with_keyword_hints': 414,
 'chunks_with_image_description': 414}

`chunks_with_image_description` is now **non-zero** - every chunk carries the merged document-level `image_description_text` assembled from the Image Analysis captions. The summaries and keyword hints from Lab 05 are still present too. The next cell shows the actual captions and figure thumbnails that landed.

## Step 3 — Inspect what the visual lane actually produced

In [3]:
import pandas as pd

chunks = lab.load_chunks(job)
with_img_desc = [c for c in chunks if (c.get('image_description_text') or '').strip()]
with_thumbs = [c for c in chunks if c.get('image_evidence')]
print('chunks carrying merged image_description_text:', len(with_img_desc))
print('chunks with figure thumbnails (image_evidence):', len(with_thumbs))

# The merged visual summary is document-level, so it is identical across chunks -
# print it once to see the Image Analysis captions that were stitched together.
print('\nDocument-level visual summary (Image Analysis captions):')
print(' ', (with_img_desc[0]['image_description_text'] if with_img_desc else '(none)'))

# Per-page figure thumbnails attached by the parser.
pd.DataFrame([
    {
        'section': ' > '.join(c.get('section_path') or []) or '(root)',
        'pages': c.get('page_numbers'),
        'figures_on_page': len(c.get('image_evidence') or []),
    }
    for c in with_thumbs[:8]
])

chunks carrying merged image_description_text: 414
chunks with figure thumbnails (image_evidence): 125

Document-level visual summary (Image Analysis captions):
  a large red and white building a construction site with cranes and buildings a high angle view of a construction site a high angle view of a port chart shape


,section,pages,figures_on_page
0,Deep Excavation Design and Construction,[1],1
1,Foreword,[4],1
2,Foreword,[4],1
3,Chany Will,[4],1
4,2.1.2 Groundwater Conditions,"[17, 18]",4
5,GIF03 (Standpipe),[18],4
6,Fill,[18],4
7,Elevation (mPD),[18],4


## Step 4 — Hybrid vs Agentic on a visual question

**Hybrid** now benefits directly: the merged `image_description_text` is part of every canonical chunk, so visual vocabulary (“construction site”, “chart”) is searchable. **Agentic** can additionally consult the Search-managed enrichment index, where the *full* per-image caption and OCR collections live, and tends to assemble more cross-section citations.

In [4]:
QV = 'What do the figures on ground settlement and wall deflection show, and what visual evidence supports the answer?'

print('--- HYBRID (canonical chunk index, now carries merged captions) ---')
resp_hybrid = lab.ask(QV, job=job, retrieval_mode='hybrid', record_as='lab06_visual_hybrid')
lab.show_answer(resp_hybrid, max_citations=4)

--- HYBRID (canonical chunk index, now carries merged captions) ---


[retrieval_mode=hybrid | scoring_profile=default | citations=7]

However, this method only calculates the lateral deflection of the embedded wall and estimation of ground settlement is usually based on empirical correlations. Lui & Yau (1995) reported the back analysis of the basement excavation at the Dragon Centre, Kowloon. [1]

Supporting evidence:
- ...Installation from Case Study Data Typical Profiles of Wall Deflection and Adjacent Ground Deformation Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data Ground Settlement due to Excavation in Coarse Grained Soils Total Ground Settlement against Maximum Excavation Depth from Case Study Data for Projects. [5]
- ...his method does not provide direct prediction of ground settlement caused by the excavation and empirical correlations are then adopted to derive the ground settlementThe mobilisation of wall friction has a significant effect on lateral earth pressure and deflection of the wall, and such b

In [5]:
print('--- AGENTIC (also reads the enrichment index: full captions + OCR) ---')
resp_agentic = lab.ask(QV, job=job, retrieval_mode='agentic', record_as='lab06_visual_agentic')
lab.show_answer(resp_agentic, max_citations=6)

--- AGENTIC (also reads the enrichment index: full captions + OCR) ---


[retrieval_mode=agentic | scoring_profile=default | citations=8]

- The figures show that **ground settlement and wall deflection are related, but not always strongly or perfectly**. [1][2]
- The document says **for small deformations** there is **no apparent correlation** between maximum ground settlement and lateral wall deflection. [3]
- **For larger deformations**, the ground settlement is **usually about 50% of the maximum lateral wall deflection**, with an exception noted for the Tsuen Wan West Station case. [3][4]
- The visual evidence supporting this is **Figure 8.4**, titled **“Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data”**, which plots the relationship using case study data and shows the settlement-to-deflection ratio ranges. [2][5]
- The related diagram captions also show the underlying wall movement and settlement pattern in **Figure 8.3, “Typical Profiles of Wall Deflection and Adjacent Ground Deformation,”** including cantilever

## Takeaways

- With the output field mappings pointed at the per-image nodes and drilled into `captions/*/text`, `OcrSkill` and `ImageAnalysisSkill` now produce **real retrievable text** - the enrichment index holds the full per-image caption + OCR collections, and a merged summary lands on every canonical chunk.
- **Verify** the signal: inspect the captions and `image_evidence` thumbnails rather than assuming. Caption quality is document-dependent - engineering diagrams give generic captions and sparse OCR.
- Hybrid gains the merged visual summary; Agentic additionally taps the full caption/OCR collections in the enrichment index.
- `ENABLE_IMAGE_UNDERSTANDING=true` is a separate, optional lane that adds richer **per-figure Foundry vision captions** (at extra cost) on top of the built-in Image Analysis signal.

Next: **Lab 07** focuses on agentic retrieval mechanics (planning, subqueries, grounded synthesis).